In [0]:
%pip install nemosis

In [0]:
# Import necessary libraries
import os
from nemosis import dynamic_data_compiler
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
nsw_dispatch_5min = spark.table("energy_endava_193.default.nsw_dispatch_5min")
nsw_dictionary = spark.table("energy_endava_193.default.nsw_dictionary")
nsw_dictionary_second = spark.table("energy_endava_193.default.nsw_dictionary_second")
nsw_demand = spark.table("energy_endava_193.default.nsw_demand")
nsw_scada = spark.table("energy_endava_193.default.nsw_scada")
# Read "PU and Scheduled Loads" sheet from Excel file using pandas, then convert to Spark DataFrame
nsw_registration_pd = pd.read_excel("/Workspace/Users/quangthanhdong04.au@gmail.com/energy-endava-193/src/energy_endava_193_etl/transformations/dataset/NEM Registration and Exemption List.xlsx", sheet_name="PU and Scheduled Loads")
# Convert all object columns to string to handle mixed types
for col in nsw_registration_pd.select_dtypes(include=['object']).columns:
    nsw_registration_pd[col] = nsw_registration_pd[col].astype(str)
nsw_registration = spark.createDataFrame(nsw_registration_pd)

# Read co2eii_available_generators.csv using pandas
# The file structure: Line 0 is metadata, Line 1 has column headers (starting with 'I'), Line 2+ are data rows (starting with 'D')
raw_generators_pd = pd.read_csv("/Workspace/Users/quangthanhdong04.au@gmail.com/energy-endava-193/src/energy_endava_193_etl/transformations/dataset/co2eii_available_generators.csv", header=None, skiprows=1)
# Use first row (line 1 of file) as header, skipping the first column which is the row type indicator
header = raw_generators_pd.iloc[0, 1:].tolist()
data_generators_pd = raw_generators_pd.iloc[1:, 1:].copy()  # Skip first column (row type) and first row (header)
data_generators_pd.columns = header
data_generators = spark.createDataFrame(data_generators_pd)

In [0]:
display(nsw_registration_pd)

In [0]:
display(nsw_dictionary_second)

In [0]:
# Mapping qualified generators with their respective "Fuel Source - Primary"
# Using 2 different tables: nsw_registration and data_generators
# The generators whose DUIDs are not present in these tables are individually researched and added into the nsw_dictionary table.
from pyspark.sql.functions import trim, upper, lower, when, col, isnull, lit, row_number
from pyspark.sql.window import Window

# Prepare nsw_registration mapping DataFrame with standardized DUID and relevant columns (case-insensitive, trimmed)
nsw_reg_fuel = nsw_registration.select(
    trim(upper(col("DUID"))).alias("DUID"),
    trim(upper(col("Fuel Source - Primary"))).alias("Fuel Source - Primary"),
    trim(upper(col("Fuel Source - Descriptor"))).alias("Fuel Source - Descriptor"),
    trim(upper(col("Technology Type - Primary"))).alias("Technology Type - Primary"),
    trim(upper(col("Technology Type - Descriptor"))).alias("Technology Type - Descriptor"),
    col("Reg Cap generation (MW)"),
    col("Max Cap generation (MW)"),
    col("Max ROC/Min generation"),
    col("Reg Cap consumption (MW)"),
    col("Max Cap consumption (MW)"),
    col("Max ROC/Min consumption"),
    col("Maximum storage capacity ")
).distinct()

# Standardize DUID in nsw_dictionary for join (case-insensitive)
nsw_dict_std = nsw_dictionary.withColumn("DUID", trim(upper(col("DUID"))))

# Left join nsw_dictionary with nsw_registration fuel info
nsw_dictionary_mapped = nsw_dict_std.join(
    nsw_reg_fuel,
    on="DUID",
    how="left"
)

# Standardize DUID in data_generators for join (case-insensitive, trimmed)
data_generators_std = data_generators.withColumn("DUID", trim(upper(col("DUID"))))

# Join nsw_dictionary_mapped with data_generators to get CO2E_ENERGY_SOURCE (case-insensitive, trimmed)
nsw_dictionary_mapped = nsw_dictionary_mapped.join(
    data_generators_std.select("DUID", trim(upper(col("CO2E_ENERGY_SOURCE"))).alias("CO2E_ENERGY_SOURCE")),
    on="DUID",
    how="left"
)

# Update "Fuel Source - Primary" for DUIDs starting with "TUMUT" or "TUMT" (case-insensitive)
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "Fuel Source - Primary",
    when(
        (upper(col("DUID")).startswith("TUMUT")) | (upper(col("DUID")).startswith("TUMT")) | (upper(col("DUID")) == "KIDSPHG2"),
        "HYDRO"
    ).otherwise(col("Fuel Source - Primary"))
).withColumn(
    "Fuel Source - Descriptor",
    when(
        (upper(col("DUID")).startswith("TUMUT")) | (upper(col("DUID")).startswith("TUMT")) | (upper(col("DUID")) == "KIDSPHG2"),
        "WATER"
    ).otherwise(col("Fuel Source - Descriptor"))
)

# Map CO2E_ENERGY_SOURCE to "Fuel Source - Primary" only for NA values (case-insensitive)
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "Fuel Source - Primary",
    when(
        isnull(col("Fuel Source - Primary")),
        col("CO2E_ENERGY_SOURCE")
    ).otherwise(col("Fuel Source - Primary"))
)

nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "Fuel Source - Descriptor",
    when(
        isnull(col("Fuel Source - Descriptor")),
        col("CO2E_ENERGY_SOURCE")
    ).otherwise(col("Fuel Source - Descriptor"))
)

# Map the generators without information in either tables with Fuel Source - Primary == "Battery Storage" and Fuel Source - Descriptor == "Grid" (case-insensitive)
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "Fuel Source - Primary",
    when(
        (upper(col("DISPATCHTYPE")) != "LOAD") &
        (upper(col("STARTTYPE")) != "NOT DISPATCHED") &
        (col("Fuel Source - Primary").isNull()) &
        (~upper(col("DUID")).isin(["DG_NSW1", "DG_VIC1", "DG_QLD1", "DG_SA1", "DG_TAS1", "KIDSPHG2"])),
        lit("BATTERY STORAGE")
    ).otherwise(col("Fuel Source - Primary"))
).withColumn(
    "Fuel Source - Descriptor",
    when(
        (upper(col("DISPATCHTYPE")) != "LOAD") &
        (upper(col("STARTTYPE")) != "NOT DISPATCHED") &
        (col("Fuel Source - Primary") == "BATTERY STORAGE") &
        (~upper(col("DUID")).isin(["DG_NSW1", "DG_VIC1", "DG_QLD1", "DG_SA1", "DG_TAS1", "KIDSPHG2"])),
        lit("GRID")
    ).otherwise(col("Fuel Source - Descriptor"))
).withColumn(
    "Fuel Source - Primary",
    when(
        (upper(col("DUID")).isin(["DG_NSW1", "DG_VIC1", "DG_QLD1", "DG_SA1", "DG_TAS1"])),
        lit("AGGREGATED")
    ).otherwise(col("Fuel Source - Primary"))
).withColumn(
    "Fuel Source - Descriptor",
    when(
        (upper(col("DUID")).isin(["DG_NSW1", "DG_VIC1", "DG_QLD1", "DG_SA1", "DG_TAS1"])),
        lit("AGGREGATED")
    ).otherwise(col("Fuel Source - Descriptor"))
).withColumn(
    "Technology Type - Primary",
    when(
        (col("Fuel Source - Descriptor") == "GRID") &
        (col("Technology Type - Primary").isNull()),
        lit("STORAGE")
    ).otherwise(col("Technology Type - Primary"))
).withColumn(
    "Technology Type - Descriptor",
    when(
        (col("Fuel Source - Descriptor") == "GRID") &
        (col("Technology Type - Descriptor").isNull()),
        lit("BATTERY")
    ).otherwise(col("Technology Type - Descriptor"))
)

# Rename columns to replace hyphens with underscores for Delta Lake compatibility
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumnRenamed("Fuel Source - Primary", "FUELSOURCEPRIMARY") \
    .withColumnRenamed("Fuel Source - Descriptor", "FUELSOURCEDESCRIPTOR") \
    .withColumnRenamed("Technology Type - Primary", "TECHNOLOGYTYPEPRIMARY") \
    .withColumnRenamed("Technology Type - Descriptor", "TECHNOLOGYTYPEDESCRIPTOR") \
    .withColumnRenamed("Reg Cap generation (MW)", "REGCAPGENERATION") \
    .withColumnRenamed("Max Cap generation (MW)", "MAXCAPGENERATION") \
    .withColumnRenamed("Max ROC/Min generation", "MAXROC_MIN_GENERATION") \
    .withColumnRenamed("Reg Cap consumption (MW)", "REGCAPCONSUMPTION") \
    .withColumnRenamed("Max Cap consumption (MW)", "MAXCAPCONSUMPTION") \
    .withColumnRenamed("Max ROC/Min consumption", "MAXROC_MIN_CONSUMPTION") \
    .withColumnRenamed("Maximum storage capacity ", "MAXSTORAGECAPACITY")

# ========================================================================
# ENRICH WITH DUDETAIL CAPACITY DATA
# ========================================================================
print("\n" + "="*70)
print("Enriching with DUDETAIL capacity data")
print("="*70)

# Deduplicate nsw_dictionary_second by DUID, keeping highest VERSIONNO
window_spec = Window.partitionBy("DUID").orderBy(col("VERSIONNO").desc())

nsw_dictionary_second_latest = nsw_dictionary_second.withColumn(
    "row_num", 
    row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

print(f"\nℹ️  Original nsw_dictionary_second rows: {nsw_dictionary_second.count():,}")
print(f"ℹ️  Deduplicated rows (latest VERSIONNO per DUID): {nsw_dictionary_second_latest.count():,}")

# Select only the columns we need from nsw_dictionary_second
nsw_dict_second_cols = nsw_dictionary_second_latest.select(
    "DUID",
    "REGISTEREDCAPACITY",
    "AGCCAPABILITY", 
    "MAXCAPACITY",
    "NORMALLYONFLAG"
)

# Join with nsw_dictionary_mapped to add the new columns
nsw_dictionary_mapped = nsw_dictionary_mapped.join(
    nsw_dict_second_cols,
    on="DUID",
    how="left"
)

print(f"✅ Added columns: REGISTEREDCAPACITY, AGCCAPABILITY, MAXCAPACITY, NORMALLYONFLAG")

# Refine FUELSOURCEDESCRIPTOR
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "FUELSOURCEDESCRIPTOR",
    when(
        upper(trim(col("FUELSOURCEDESCRIPTOR"))).isin(
            "NATURAL GAS / FUEL OIL",
            "NATURAL GAS / DIESEL",
            "NATRUAL GAS/ DIESEL",
            "NATURAL GAS (PIPELINE)"
        ),
        "NATURAL GAS"
    ).when(
        upper(trim(col("FUELSOURCEDESCRIPTOR"))) == "KEROSENE - NON AVIATION",
        "KEROSENE"
    ).when(
        upper(trim(col("FUELSOURCEDESCRIPTOR"))) == "DIESEL OIL",
        "DIESEL"
    ).otherwise(col("FUELSOURCEDESCRIPTOR"))
)

# Refine FUELSOURCEPRIMARY
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "FUELSOURCEPRIMARY",
    when(
        upper(trim(col("FUELSOURCEPRIMARY"))).isin(
            "DIESEL OIL",
            "NATURAL GAS (PIPELINE)",
            "KEROSENE - NON AVIATION",
            "BLACK COAL"
        ),
        "FOSSIL"
    ).otherwise(col("FUELSOURCEPRIMARY"))
)

# Refine TECHNOLOGYTYPEPRIMARY
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
  "TECHNOLOGYTYPEPRIMARY",
  when(
    upper(trim(col("FUELSOURCEPRIMARY"))) == "FOSSIL",
    "COMBUSTION"
  ).when(
    upper(trim(col("DUID"))) == "KIDSPHG2",
    "RENEWABLE"
  ).when(
    upper(trim(col("FUELSOURCEDESCRIPTOR"))) == "AGGREGATED",
    "AGGREGATED"
  ).otherwise(col("TECHNOLOGYTYPEPRIMARY"))
)

# Refine TECHNOLOGYTYPEDESCRIPTOR
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
  "TECHNOLOGYTYPEDESCRIPTOR",
  when(
    upper(trim(col("DUID"))) == "KIDSPHG2",
    "PUMP STORAGE"
  ).when(
    upper(trim(col("TECHNOLOGYTYPEDESCRIPTOR"))) == "PHOTOVOLTALIC TRACKING FLAT PANEL",
    "PHOTOVOLTAIC TRACKING FLAT PANEL"
  ).when(
        upper(trim(col("TECHNOLOGYTYPEDESCRIPTOR"))).isin(
            "OCGT",
            "OPEN CYCLE GAS TURBINE (OCGT)"
        ) | upper(trim(col("FUELSOURCEDESCRIPTOR"))).isin(
          "KEROSENE",
          "DIESEL"
        ),
        "OPEN CYCLE GAS TURBINES (OCGT)"
  ).when(
    (upper(trim(col("DISPATCHTYPE"))) == "BIDIRECTIONAL") & (col("TECHNOLOGYTYPEDESCRIPTOR") == "BATTERY"),
    "BATTERY AND INVERTER"
  ).when(
    upper(trim(col("DUID"))).isin(
      "LD01",
      "LD02",
      "LD03",
      "LD04",
      "TORRA3"
    ),
    "STEAM SUB-CRITICAL"
  ).when(
    upper(trim(col("FUELSOURCEDESCRIPTOR"))) == "AGGREGATED",
    "AGGREGATED"
  ).otherwise(col("TECHNOLOGYTYPEDESCRIPTOR"))
)

# Refine STARTTYPE
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
  "STARTTYPE",
  when(
    (col("STARTTYPE").isNull()),
    "NOT DISPATCHED"
  ).otherwise(col("STARTTYPE"))
)

# Drop specified columns
nsw_dictionary_mapped = nsw_dictionary_mapped.drop(
    "CONNECTIONPOINTID",
    "STATIONID",
    "PARTICIPANTID",
    "LASTCHANGED"
)

# Save nsw_dictionary_mapped to Delta Lake
nsw_dictionary_mapped.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("energy_endava_193.default.nsw_dictionary_mapped")
print("✅ nsw_dictionary_mapped saved to Delta Lake")

print(f"📊 Final row count: {nsw_dictionary_mapped.count():,}")
print("="*70)

display(nsw_dictionary_mapped)

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define explicit schema
schema = StructType([
    StructField("TECHNOLOGYTYPEDESCRIPTOR", StringType(), False),
    StructField("MIN_STABLE_GENERATION", IntegerType(), False),
    StructField("STARTUP_TIME", IntegerType(), False),
    StructField("MIN_UPTIME", IntegerType(), False),
    StructField("MIN_DOWNTIME", IntegerType(), False),
    StructField("MIN_RAMP_RATE_PROXY", IntegerType(), False),
    StructField("MAX_RAMP_RATE_PROXY", IntegerType(), False)
])

# Create rows for each technology descriptor
tech_rows = [
    Row("BATTERY", 0, 0, 0, 0, 100, 100),
    Row("BATTERY AND INVERTER", 0, 0, 0, 0, 100, 100),
    Row("COMBINED CYCLE GAS TURBINE (CCGT)", 46, 145, 240, 240, 3, 5),
    Row("COMPRESSION RECIPROCATING ENGINE", 40, 10, 15, 15, 20, 40),
    Row("HYDRO - GRAVITY", 5, 5, 0, 0, 20, 50),
    Row("OPEN CYCLE GAS TURBINES (OCGT)", 50, 30, 30, 30, 10, 20),
    Row("PHOTOVOLTAIC FLAT PANEL", 0, 0, 0, 0, 101, 101),
    Row("PHOTOVOLTAIC TRACKING FLAT PANEL", 0, 0, 0, 0, 101, 101),
    Row("PUMP STORAGE", 10, 10, 60, 60, 15, 25),
    Row("RUN OF RIVER", 0, 0, 0, 0, 102, 102),
    Row("STEAM SUB-CRITICAL", 40, 420, 720, 720, 1, 2),
    Row("STEAM SUPER CRITICAL", 40, 420, 720, 720, 1, 3),
    Row("WIND - ONSHORE", 0, 0, 0, 0, 101, 101),
    Row("AGGREGATED", 0, 0, 0, 0, 101, 101),
]

# Create Spark DataFrame with explicit schema
nsw_generators_constraints = spark.createDataFrame(tech_rows, schema)

# Write to Delta Lake table in energy_endava_193.default
nsw_generators_constraints.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("energy_endava_193.default.nsw_generators_constraints")

print("✅ nsw_generators_constraints saved to Delta Lake")

display(nsw_generators_constraints)

In [0]:
# Split nsw_dictionary_mapped into nsw_generators and nsw_loads based on DISPATCHTYPE
nsw_generators = spark.table("energy_endava_193.default.nsw_dictionary_mapped") \
    .filter(upper(col("DISPATCHTYPE")).isin("BIDIRECTIONAL", "GENERATOR"))

# Left join nsw_generators with nsw_generators_constraints on TECHNOLOGYTYPEDESCRIPTOR
nsw_generators_constraints = spark.table("energy_endava_193.default.nsw_generators_constraints")
nsw_generators = nsw_generators.join(
    nsw_generators_constraints,
    on="TECHNOLOGYTYPEDESCRIPTOR",
    how="left"
)

# Split nsw_generators into scheduled and non-scheduled based on SCHEDULE_TYPE
nsw_generators_scheduled = nsw_generators.filter(upper(col("SCHEDULE_TYPE")) != "NON-SCHEDULED")
nsw_generators_non_scheduled = nsw_generators.filter(upper(col("SCHEDULE_TYPE")) == "NON-SCHEDULED")

nsw_generators_scheduled.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("energy_endava_193.default.nsw_generators_scheduled")
print("✅ nsw_generators_scheduled saved to Delta Lake")

nsw_generators_non_scheduled.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("energy_endava_193.default.nsw_generators_non_scheduled")
print("✅ nsw_generators_non_scheduled saved to Delta Lake")

nsw_loads = spark.table("energy_endava_193.default.nsw_dictionary_mapped") \
    .filter(upper(col("DISPATCHTYPE")).isin("BIDIRECTIONAL", "LOAD"))

nsw_loads.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("energy_endava_193.default.nsw_loads")

print("✅ nsw_loads saved to Delta Lake")

display(nsw_generators_scheduled)
display(nsw_generators_non_scheduled)
display(nsw_loads)

In [0]:
nsw_dispatch_demand = nsw_demand.join(
    nsw_dispatch_5min,
    on=["SETTLEMENTDATE", "REGIONID", "INTERVENTION"],
    how="inner"
)

# Save nsw_dispatch_demand_df to Delta Lake
nsw_dispatch_demand.write.mode("overwrite").saveAsTable("energy_endava_193.default.nsw_dispatch_demand")
print("✅ nsw_dispatch_demand saved to Delta Lake")

display(nsw_dispatch_demand)

In [0]:
# Find a specific day within November 2022 for a peak TOTALDEMAND
from pyspark.sql.functions import year, month, dayofmonth, max, col

# Filter for November 2024
nov_2022_df = nsw_dispatch_demand.filter(
    (year(col("SETTLEMENTDATE")) == 2022) &
    (month(col("SETTLEMENTDATE")) == 11)
)

# Find the day with the highest TOTALDEMAND peak
peak_day_df = nov_2022_df.groupBy(dayofmonth(col("SETTLEMENTDATE")).alias("day")) \
    .agg(max(col("TOTALDEMAND")).alias("peak_demand")) \
    .orderBy(col("peak_demand").desc()) \
    .limit(1)

# Get the day value
peak_day = peak_day_df.collect()[0]["day"]

# Create the dataframe for all times in the peak day
nsw_dispatch_demand_peak_2022 = nov_2022_df.filter(dayofmonth(col("SETTLEMENTDATE")) == peak_day)

# Save nsw_dispatch_demand_peak_2022 to Delta Lake
nsw_dispatch_demand_peak_2022.write.mode("overwrite").saveAsTable("energy_endava_193.default.nsw_dispatch_demand_peak_2022")
print("✅ nsw_dispatch_demand_peak_2022 saved to Delta Lake")

display(nsw_dispatch_demand_peak_2022)

In [0]:
from pyspark.sql.functions import to_date, col, lit

nsw_scada_peak_2022 = nsw_scada.filter(
    to_date(col("SETTLEMENTDATE")) == lit("2022-11-01")
)

# Save nsw_scada_peak_2022 to Delta Lake
nsw_scada_peak_2022.write.mode("overwrite").saveAsTable("energy_endava_193.default.nsw_scada_peak_2022")
print("✅ nsw_scada_peak_2022 saved to Delta Lake")

display(nsw_scada_peak_2022)

In [0]:
# Read the NEM Registration and Exemption List
import pandas as pd

nsw_registration_pd = pd.read_excel(
    "/Workspace/Users/quangthanhdong04.au@gmail.com/energy-endava-193/src/energy_endava_193_etl/transformations/dataset/NEM Registration and Exemption List.xlsx", 
    sheet_name="PU and Scheduled Loads"
)

# Select only the columns we used in the preprocessing
columns_used = [
    "DUID",
    "Fuel Source - Primary",
    "Fuel Source - Descriptor",
    "Technology Type - Primary",
    "Technology Type - Descriptor",
    "Reg Cap generation (MW)",
    "Max Cap generation (MW)",
    "Reg Cap consumption (MW)",
    "Max Cap consumption (MW)"
]

sample_df = nsw_registration_pd[columns_used].head(5)

display(sample_df)